# Treasury Lock Validation — Pucci (2019)

**Reference:** M. Pucci, *Hedging the Treasury Lock*, Banca IMI, 2019. SSRN 3386521.

Visual validation of the pricebook `treasury_lock` module against the paper's results.

## Contents
1. T-Lock payoff vs forward proxy (overhedge)
2. Overhedge error vs yield move
3. Delta and gamma profiles
4. Forward price sensitivity to repo
5. Roll P&L surface

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "python"))

import numpy as np
import matplotlib.pyplot as plt
from pricebook.bond_yield import bond_price_from_yield, bond_risk_factor
from pricebook.bond_forward import forward_price_repo
from pricebook.treasury_lock import (
    tlock_delta,
    tlock_gamma,
    overhedge_bound,
    roll_pnl,
)

# Standard 10Y benchmark
COUPON = 0.03
ALPHAS = [0.5] * 20
TIMES = [0.5 * (i + 1) for i in range(20)]
T_MAT = 10.0
L = 0.03  # locked yield (ATM)

plt.rcParams.update({"figure.figsize": (10, 6), "font.size": 12})
print("Setup complete.")

## 1. T-Lock Payoff vs Forward Proxy (Paper Fig 1)

The exact T-Lock payoff is $g(y) = -D_y[P](y)(y - L)$ (RiskFactor times yield move).

The forward proxy is $P(L) - P(y)$ (price gap).

The proxy **strictly overhedges**: $P(L) - P(y) \geq g(y)$ for all $y \geq 0$, with equality only at $y = L$. This is because the bond price is convex in yield (Proposition 1).

In [ ]:
yields = np.linspace(0.001, 0.08, 200)
P_L = bond_price_from_yield(COUPON, ALPHAS, L)

exact_payoff = [bond_risk_factor(COUPON, ALPHAS, y) * (y - L) for y in yields]
proxy_payoff = [P_L - bond_price_from_yield(COUPON, ALPHAS, y) for y in yields]
bond_prices = [bond_price_from_yield(COUPON, ALPHAS, y) for y in yields]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(yields * 100, exact_payoff, "b-", lw=2, label="Exact T-Lock payoff")
ax1.plot(yields * 100, proxy_payoff, "r--", lw=2, label="Forward proxy (overhedge)")
ax1.axvline(L * 100, color="gray", ls=":", alpha=0.5, label=f"L = {L*100:.0f}%")
ax1.axhline(0, color="k", lw=0.5)
ax1.set_xlabel("Yield (%)")
ax1.set_ylabel("Payoff (per unit notional)")
ax1.set_title("T-Lock Payoff vs Forward Proxy")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(bond_prices, exact_payoff, "b-", lw=2, label="Exact T-Lock")
ax2.plot(bond_prices, proxy_payoff, "r--", lw=2, label="Forward proxy")
ax2.axvline(P_L, color="gray", ls=":", alpha=0.5, label=f"P(L) = {P_L:.4f}")
ax2.axhline(0, color="k", lw=0.5)
ax2.set_xlabel("Bond Price")
ax2.set_ylabel("Payoff")
ax2.set_title("Payoff vs Bond Price (Paper Fig 1)")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Overhedge Error vs Yield Move

The Taylor remainder $R_1(y) = P(y) - P(L) - D_y[P](L)(y-L) \geq 0$ (Proposition 1).

Bounded by $|R_1| \leq \frac{1}{2} M (y-L)^2$ where $M = (t_n-t)^2 + c \sum \alpha_i (t_i-t)^2$ (Eq 10-11).

For a 10Y benchmark with $c \sim 3\%$, Pucci computes $M \sim 150$.

In [ ]:
dy = np.linspace(-0.03, 0.03, 200)
R1 = []
bound = []

for d in dy:
    y = L + d
    if y <= 0:
        R1.append(np.nan)
        bound.append(np.nan)
        continue
    P_y = bond_price_from_yield(COUPON, ALPHAS, y)
    proxy = P_L - P_y
    exact = bond_risk_factor(COUPON, ALPHAS, y) * (y - L)
    R1.append(proxy - exact)
    bound.append(overhedge_bound(COUPON, ALPHAS, TIMES, T_MAT, d))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(dy * 100, R1, "b-", lw=2, label="$R_1(y)$ (actual overhedge)")
ax.plot(dy * 100, bound, "r--", lw=2, label="$\\frac{1}{2} M (y-L)^2$ bound")
ax.axvline(0, color="gray", ls=":", alpha=0.5)
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("Yield change $y - L$ (% points)")
ax.set_ylabel("Overhedge error")
ax.set_title("Overhedge Error: Always Non-Negative, Bounded by Convexity")
ax.legend()
ax.grid(alpha=0.3)

M = T_MAT**2 + COUPON * sum(a * t**2 for a, t in zip(ALPHAS, TIMES))
ax.text(0.02, 0.85, f"$M = {M:.1f}$\n10Y, c = {COUPON*100:.0f}%",
        transform=ax.transAxes, fontsize=11,
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.tight_layout()
plt.show()

## 3. Delta and Gamma Profiles

**Delta** (Eq 14-15): $\Delta = D_P[g] < 0$ for a long T-Lock — short the bond's price.

**Gamma** (Eq 16-17): $\Gamma < 0$ near $L$ — negative convexity. The proxy earns gamma P&L: $\text{P\&L} \approx -\frac{1}{2}\Gamma(\Delta P)^2 \geq 0$ (Eq 20).

In [ ]:
yields_greek = np.linspace(0.005, 0.08, 200)

deltas = [tlock_delta(COUPON, ALPHAS, TIMES, T_MAT, y, L, direction=1) for y in yields_greek]
gammas = [tlock_gamma(COUPON, ALPHAS, TIMES, T_MAT, y, L, direction=1) for y in yields_greek]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(yields_greek * 100, deltas, "b-", lw=2)
ax1.axhline(0, color="k", lw=0.5)
ax1.axvline(L * 100, color="gray", ls=":", alpha=0.5, label=f"L = {L*100:.0f}%")
ax1.set_xlabel("Yield (%)")
ax1.set_ylabel("Delta $D_P[g]$")
ax1.set_title("Delta: Negative (Short Bond Price)")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(yields_greek * 100, gammas, "r-", lw=2)
ax2.axhline(0, color="k", lw=0.5)
ax2.axvline(L * 100, color="gray", ls=":", alpha=0.5, label=f"L = {L*100:.0f}%")
ax2.set_xlabel("Yield (%)")
ax2.set_ylabel("Gamma $D^2_P[g]$")
ax2.set_title("Gamma: Negative (Proxy Earns Gamma P&L)")
ax2.legend()
ax2.grid(alpha=0.3)

# Annotate gamma at L
g_at_L = tlock_gamma(COUPON, ALPHAS, TIMES, T_MAT, L, L, direction=1)
ax2.annotate(f"$\\Gamma(L)$ = {g_at_L:.4f}", xy=(L*100, g_at_L),
             xytext=(L*100 + 1.5, g_at_L + 0.001),
             arrowprops=dict(arrowstyle="->"), fontsize=10)

plt.tight_layout()
plt.show()

## 4. Forward Price Sensitivity to Repo

From Eq (25): higher repo rate $\Rightarrow$ richer forward price $\Rightarrow$ lower forward IRR $\Rightarrow$ cheaper long T-Lock.

Specialness on the on-the-run benchmark makes the long T-Lock significantly more expensive.

In [ ]:
import math

mkt_price = bond_price_from_yield(COUPON, ALPHAS, L)
tau = 0.5  # 6M expiry

repo_rates = np.linspace(-0.01, 0.06, 200)
fwd_prices = [forward_price_repo(mkt_price, r, tau, COUPON, [], []) for r in repo_rates]

K = bond_price_from_yield(COUPON, ALPHAS, L)
tlock_pvs = [math.exp(-0.03 * tau) * (K - fwd) for fwd in fwd_prices]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(repo_rates * 100, fwd_prices, "b-", lw=2)
ax1.axhline(mkt_price, color="gray", ls=":", alpha=0.5, label=f"Spot = {mkt_price:.4f}")
ax1.set_xlabel("Repo Rate (%)")
ax1.set_ylabel("Forward Price")
ax1.set_title("Forward Price vs Repo Rate")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(repo_rates * 100, tlock_pvs, "r-", lw=2)
ax2.axhline(0, color="k", lw=0.5)
ax2.set_xlabel("Repo Rate (%)")
ax2.set_ylabel("Long T-Lock PV")
ax2.set_title("Long T-Lock Value vs Repo (Eq 25: Higher Repo = Cheaper)")
ax2.grid(alpha=0.3)

ax2.annotate("Higher repo\n= cheaper long T-Lock", xy=(4, tlock_pvs[150]),
             fontsize=10, ha="center",
             bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

plt.tight_layout()
plt.show()

## 5. Roll P&L Surface

From Eq (31): roll P&L depends on coupon change $(\hat{c} - c)$ and yield change $(\hat{R} - R)$.

Vanishes when $\hat{c} = c$ AND $\hat{R} = R$ (no change at re-issue).

In [ ]:
dc_range = np.linspace(-0.01, 0.01, 50)   # coupon change
dR_range = np.linspace(-0.005, 0.005, 50)  # yield change
DC, DR = np.meshgrid(dc_range, dR_range)

R_base = 0.035
roll_surface = np.zeros_like(DC)

for i in range(DC.shape[0]):
    for j in range(DC.shape[1]):
        roll_surface[i, j] = roll_pnl(
            COUPON, COUPON + DC[i, j],
            R_base, R_base + DR[i, j],
            L, ALPHAS, TIMES, T_MAT,
        )

fig, ax = plt.subplots(figsize=(10, 7))
c = ax.contourf(DC * 100, DR * 100, roll_surface * 100, levels=20, cmap="RdBu_r")
ax.contour(DC * 100, DR * 100, roll_surface * 100, levels=[0], colors="k", linewidths=2)
fig.colorbar(c, ax=ax, label="Roll P&L (% of notional)")
ax.set_xlabel("Coupon change $\\hat{c} - c$ (% points)")
ax.set_ylabel("Yield change $\\hat{R} - R$ (% points)")
ax.set_title("Roll P&L Surface (Eq 31) — Zero contour in black")
ax.axhline(0, color="gray", ls=":", alpha=0.3)
ax.axvline(0, color="gray", ls=":", alpha=0.3)
ax.plot(0, 0, "ko", ms=8, label="No change (P&L = 0)")
ax.legend()
plt.tight_layout()
plt.show()